# Example of initial polulating of Memgraph

## Loading the dependencies

In [1]:
import os
from neo4j import GraphDatabase, AsyncGraphDatabase
import pandas as pd
import requests

from functionality.agent import load_env

In [2]:
load_env()
URI = os.getenv("MEMGRAPH_URI")
AUTH = (os.getenv("MEMGRAPH_USER"), os.getenv("MEMGRAPH_PASSWORD"))
url = "http://localhost:9090/vectors" # Weaviate vectorizer container URL

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as client:
    # Check the connection
    client.verify_connectivity()

## Loading the data

Let's assume that we have a CSV file with the following structure:

- title
- text
- category

In [ ]:
df = pd.read_csv("path/to/file.csv", sep="$")
df.head()

In [ ]:
df.info()

## (Optional) Creating ENUM

To increase the memory-efficency and search speed, let's create an ENUM based on the unique categories in the CSV file.

In [14]:
# Creaty ENUM for categories
categories = df["category"].unique().tolist()
create_categories_enum_query = f"CREATE ENUM Category VALUES { set(categories) };".replace("'","")

In [ ]:
for c in categories:
    print(c)

In [ ]:
create_categories_enum_query

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH).session(database="memgraph")  as session:
    session.run(
        create_categories_enum_query)


## Loading data into Memgraph

In this example, we also have a Weaviate vectorizer set up to automatically vectorize the nodes titles. It will allow to use a similarity search on the title field.

But before we need to create the vector index in Memgraph. In scope of the existing implementation, it has a major downside - vector index requires a node Label to be known BEFORE the index creation. And the pipeline assumes that the LLM can create it's own labels based on the data. So, it's hard to predict in advance with which labels we will end. This can be resolved by forcing LLM to use a predefined set of labels, but it will reduce the flexibility, so this is a trade-off.

### Creating vector index on existing labels

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as client:
    with client.session(database="memgraph") as session:
        for category in categories:
            session.run(f"""CREATE VECTOR INDEX titleVector{category} ON :{category}(title_vector) WITH CONFIG {{"dimension": 768, "capacity": 1024 , "metric": "cos"}};""")

### Loading data

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as client:
    with client.session(database="memgraph") as session:
        for index, row in df.iterrows():
            title=f"\"{row['title']}\""
            payload = {"text": row['title']}
            resp = requests.post(url, json=payload)
            title_vector = resp.json()["vector"]
            session.run(
                query.format(category=row["category"], title=title, title_vector=title_vector) )
        result = session.run("MATCH (n) RETURN n;")
        for record in result:
            print(record)


## Test quering

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as client:
    records, summary, keys = client.execute_query("""MATCH (n {{title: '{title}'}}) RETURN n""".format(title="Sample title"))

In [ ]:
records, summary, keys

In [8]:
cypher_query_find_nodes_relationships = """MATCH (a {{title: '{node1_title}'}})-[r]->(b {{title: '{node2_title}'}}) RETURN r"""

In [ ]:
cypher_query_find_nodes_relationships

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as client:
    records, summary, keys = client.execute_query(
        cypher_query_find_nodes_relationships.format(
            node1_title="Sample title",
            node2_title="Other title",
        )
    )

In [ ]:
print(len(records))
for record in records:
    print(record)